In [1]:
import numpy as np


import requests
import re
     

In [3]:
# get raw text from internet
book = requests.get('https://www.gutenberg.org/files/35/35-0.txt')

text = book.text
print(type(text))
print(len(text))

text[:2000]

<class 'str'>
179791


'*** START OF THE PROJECT GUTENBERG EBOOK 35 ***\n\n\n\n\nThe Time Machine\n\nAn Invention\n\nby H. G. Wells\n\n\nCONTENTS\n\n I Introduction\n II The Machine\n III The Time Traveller Returns\n IV Time Travelling\n V In the Golden Age\n VI The Sunset of Mankind\n VII A Sudden Shock\n VIII Explanation\n IX The Morlocks\n X When Night Came\n XI The Palace of Green Porcelain\n XII In the Darkness\n XIII The Trap of the White Sphinx\n XIV The Further Vision\n XV The Time Traveller’s Return\n XVI After the Story\n Epilogue\n\n\n\n\n I.\n Introduction\n\n\nThe Time Traveller (for so it will be convenient to speak of him) was\nexpounding a recondite matter to us. His pale grey eyes shone and\ntwinkled, and his usually pale face was flushed and animated. The fire\nburnt brightly, and the soft radiance of the incandescent lights in the\nlilies of silver caught the bubbles that flashed and passed in our\nglasses. Our chairs, being his patents, embraced and caressed us rather\nthan submitted to b

In [5]:
# character strings to replace with space
strings2replace = [
                 '\r\n\r\nâ\x80\x9c', # new paragraph
                 'â\x80\x9c',         # open quote
                 'â\x80\x9d',         # close quote
                 '\r\n',              # new line
                 'â\x80\x94',         # hyphen
                 'â\x80\x99',         # single apostrophe
                 'â\x80\x98',         # single quote
                 '_',                 # underscore, used for stressing
                 ]

# e.g., 'â\x80\x9d'.encode('latin1').decode('utf8')


# using regular expression to remove those strings with space
for str2match in strings2replace:
    regexp = re.compile(r'%s'%str2match)
    text = regexp.sub(' ', text)

# replacing non-ascii characters
text = re.sub(r'[^\x00-\x7F]+', ' ', text)

# remove numbers
text = re.sub(r'\d+', '', text)

# making everything lowercase
text = text.lower()

text[:2000]

'*** start of the project gutenberg ebook  ***\n\n\n\n\nthe time machine\n\nan invention\n\nby h. g. wells\n\n\ncontents\n\n i introduction\n ii the machine\n iii the time traveller returns\n iv time travelling\n v in the golden age\n vi the sunset of mankind\n vii a sudden shock\n viii explanation\n ix the morlocks\n x when night came\n xi the palace of green porcelain\n xii in the darkness\n xiii the trap of the white sphinx\n xiv the further vision\n xv the time traveller s return\n xvi after the story\n epilogue\n\n\n\n\n i.\n introduction\n\n\nthe time traveller (for so it will be convenient to speak of him) was\nexpounding a recondite matter to us. his pale grey eyes shone and\ntwinkled, and his usually pale face was flushed and animated. the fire\nburnt brightly, and the soft radiance of the incandescent lights in the\nlilies of silver caught the bubbles that flashed and passed in our\nglasses. our chairs, being his patents, embraced and caressed us rather\nthan submitted to be 

### Parse text into words

In [12]:
import string
print(string.punctuation)

puncts4re = fr'[{string.punctuation}\s]'

words = re.split(puncts4re, text)
words = [item.strip() for item in words if item.strip()]

# removing single-character words
words = [item for item in words if len(item) > 1]

words[:50]

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


['start',
 'of',
 'the',
 'project',
 'gutenberg',
 'ebook',
 'the',
 'time',
 'machine',
 'an',
 'invention',
 'by',
 'wells',
 'contents',
 'introduction',
 'ii',
 'the',
 'machine',
 'iii',
 'the',
 'time',
 'traveller',
 'returns',
 'iv',
 'time',
 'travelling',
 'in',
 'the',
 'golden',
 'age',
 'vi',
 'the',
 'sunset',
 'of',
 'mankind',
 'vii',
 'sudden',
 'shock',
 'viii',
 'explanation',
 'ix',
 'the',
 'morlocks',
 'when',
 'night',
 'came',
 'xi',
 'the',
 'palace',
 'of']

In [14]:
# creating vocabulary
vocab = sorted(set(words))

# convenience variables for later
nWords = len(words)
nLex = len(vocab)

print(f'{nWords} words')
print(f' {nLex} unique tokens')

30698 words
 4589 unique tokens


### Create token dictionaries and encoder/decoder functions

In [15]:
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for i,w in enumerate(vocab)}

for i in list(word2idx.items())[0:10000:87]:
  print(i)

('abandon', 0)
('aimlessly', 87)
('apologise', 174)
('attained', 261)
('behaved', 348)
('both', 435)
('can', 522)
('cheerfully', 609)
('coat', 696)
('contents', 783)
('culminating', 870)
('delay', 957)
('dimness', 1044)
('dragging', 1131)
('edition', 1218)
('everywhere', 1305)
('facilities', 1392)
('find', 1479)
('footfall', 1566)
('furnishing', 1653)
('gold', 1740)
('hallo', 1827)
('high', 1914)
('ideas', 2001)
('inextinguishable', 2088)
('invest', 2175)
('lamp', 2262)
('likewise', 2349)
('manhood', 2436)
('minerals', 2523)
('mysteries', 2610)
('novelty', 2697)
('outbreaks', 2784)
('paws', 2871)
('plato', 2958)
('previously', 3045)
('questionings', 3132)
('reflecting', 3219)
('return', 3306)
('sandals', 3393)
('senses', 3480)
('shrinking', 3567)
('slit', 3654)
('special', 3741)
('stick', 3828)
('sudden', 3915)
('tap', 4002)
('thrice', 4089)
('treat', 4176)
('unfrozen', 4263)
('vertical', 4350)
('wearisome', 4437)
('wonderful', 4524)


In [18]:
# encoder function using for loops
def encoder(words, encode_dict):
    # initialize a vector of numerical indices
    idxs = np.zeros(len(words), dtype=int)

    # looping through the words and finding their token in the vocabulary
    for i,w in enumerate(words):
        idxs[i] = encode_dict[w]

    return idxs

# decoder function
def decoder(idxs, decode_dict):
    return ' '.join([decode_dict[i] for i in idxs])

In [19]:
# test the encoder
print(encoder(['the','time','machine'],word2idx))

# test the decoder
print(decoder([1,3,10],idx2word))

[4042 4109 2416]
abandoned abnormally absent


In [24]:
# random start location
startidx = np.random.choice(nWords)

idxs = np.arange(startidx, startidx+10)

print('Word indices:')
print(idxs), print('')

print('The words:')
wordseq = [words[i] for i in idxs]
print(wordseq), print('')

print('Token indices:')
tokenseq = encoder(wordseq, word2idx)
print(tokenseq), print('')

print('Decoded text from indices:')
decoder(tokenseq,idx2word)
     

Word indices:
[25822 25823 25824 25825 25826 25827 25828 25829 25830 25831]

The words:
['trouser', 'pocket', 'were', 'still', 'some', 'loose', 'matches', 'the', 'box', 'must']

Token indices:
[4198 2974 4454 3831 3712 2392 2459 4042  441 2604]

Decoded text from indices:


'trouser pocket were still some loose matches the box must'